# 12 · QLoRA Fine-Tuning

In plain English, **QLoRA** is the trick that lets you fine-tune a *big* language model on a *small, cheap* GPU — even a free Google Colab one. It does this by **squishing** the giant base model into a tiny 4-bit form (quantization) so it fits in memory, then training only a few small "adapter" layers on top of it (LoRA). You get most of the quality of a full fine-tune for a fraction of the memory and cost.

If you've done **notebook 11 (LoRA)**, you already know 90% of this. QLoRA is just *LoRA with a 4-bit-quantized base model*. We'll explain quantization from scratch, then run a real QLoRA fine-tune on a small free model.

> **Heads-up:** the 4-bit part of QLoRA needs an **NVIDIA GPU**. On a plain laptop CPU or Apple Silicon (M1/M2/M3) Mac it won't run. We give you a friendly **fallback** plan in that case — keep reading, all the explanations still apply.

## What you'll learn

- **Quantization** explained with everyday analogies — what "4-bit" really means and why it saves so much memory, with a tiny NumPy demo you can run on any CPU.
- **What QLoRA is**: a 4-bit quantized frozen base model + small trainable **LoRA adapters**, and why this combination is a big deal.
- The **`BitsAndBytesConfig`** settings line by line: `load_in_4bit`, `bnb_4bit_quant_type="nf4"`, `bnb_4bit_use_double_quant`, `bnb_4bit_compute_dtype`.
- A full **QLoRA recipe** on a small free model (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`): load 4-bit, `prepare_model_for_kbit_training`, attach a `LoraConfig`, train with `Trainer`, and **save the adapter**.
- The key **hyperparameters** (learning rate, epochs, batch size, gradient accumulation, sequence length, warmup) seen through a QLoRA lens.
- A practical **decision guide**: when to reach for QLoRA vs. LoRA vs. full fine-tuning.

## Why this matters for fine-tuning

A 7-billion-parameter model in normal 16-bit precision needs roughly **14 GB just to hold the weights** — and full fine-tuning needs *several times* that for gradients and optimizer state. That's far beyond a free Colab GPU (which has ~15 GB total).

**QLoRA changes the math.** By storing the frozen base model in **4-bit** (about 3.5 GB for a 7B model) and training only tiny LoRA adapters, you can fine-tune a 7B-class model on a **single small GPU** that costs nothing. This is the single biggest reason hobbyists and small teams can fine-tune large models at all today.

The crucial idea to hold onto: **QLoRA only changes *how the base model is stored and which weights you train*. Everything else — loading, tokenizing, formatting data, the `Trainer` loop, saving — is the same workflow you already know.** So this notebook is mostly "LoRA from notebook 11, plus one memory trick."

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it on Colab or a fresh environment.

> ### 🟢 Does my machine support QLoRA's 4-bit path?
> The 4-bit magic comes from a library called **`bitsandbytes`**, which currently needs an **NVIDIA GPU with CUDA**.
> - ✅ **NVIDIA GPU** (e.g. a **free Colab T4** — go to *Runtime → Change runtime type → T4 GPU*): everything in this notebook runs.
> - ❌ **CPU-only laptop** or ❌ **Apple Silicon Mac (M1/M2/M3)**: `bitsandbytes` 4-bit **will not work**. Don't worry — read all the markdown, run the CPU-safe demos, and for the actual fine-tuning use the **LoRA fallback** described later (the LoRA classifier from **notebook 11**, which runs fine on CPU).
>
> The setup cell below checks your hardware and tells you which path you're on.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install transformers datasets peft accelerate bitsandbytes torch

import torch
import numpy as np

# These two are always safe to import (CPU or GPU):
import transformers
import datasets

print("torch version:       ", torch.__version__)
print("transformers version:", transformers.__version__)
print("datasets version:    ", datasets.__version__)

# Detect hardware so we know whether the 4-bit (GPU) cells can run.
HAS_GPU = torch.cuda.is_available()
print("\nCUDA (NVIDIA GPU) available:", HAS_GPU)
if HAS_GPU:
    print("GPU name:", torch.cuda.get_device_name(0))
    print("=> You can run the full QLoRA recipe below. 🎉")
else:
    print("=> No NVIDIA GPU detected.")
    print("   The 4-bit QLoRA cells will SKIP themselves with a friendly message.")
    print("   Use the LoRA fallback (notebook 11) for hands-on training.")
# Expected on free Colab T4:  CUDA (NVIDIA GPU) available: True  /  GPU name: Tesla T4
# Expected on a Mac/CPU:      CUDA (NVIDIA GPU) available: False

## 1. What is quantization? (the rounding-prices analogy)

A neural network is just a huge pile of **numbers** (the "weights"). Normally each number is stored as a **32-bit** or **16-bit** float — a very precise decimal like `0.4718239`. Precise, but each one takes up space, and there are *billions* of them.

**Quantization** stores each number using **fewer bits** — say **4 bits**, which can only represent **16 distinct values**. So instead of `0.4718239` you store the nearest allowed value, maybe `0.5`. You lose a little precision but save a *huge* amount of memory.

**Analogy:** imagine a shopping list where every price is written to the cent: `$3.47`, `$1.92`, `$8.05`. If you **round every price to the nearest dollar** (`$3`, `$2`, `$8`), the list is much shorter to write and still roughly right for budgeting. Quantization rounds the model's "prices" (weights) to a small set of allowed values. The total is slightly off, but close enough to be useful — and it takes far less space.

Key trade-off: **fewer bits = less memory, but a small accuracy cost.** QLoRA's clever design keeps that accuracy cost tiny.

In [ ]:
# A tiny CPU-only demo of quantization. No GPU needed.
# We'll take some "weights" (random floats) and quantize them to just a few levels.

np.random.seed(0)
weights = np.random.uniform(-1.0, 1.0, size=8)   # 8 pretend model weights in [-1, 1]
print("Original weights (full precision):")
print(np.round(weights, 4))

def quantize(x, levels=16):
    """Round each value to one of `levels` evenly spaced steps between min and max."""
    lo, hi = x.min(), x.max()
    step = (hi - lo) / (levels - 1)          # size of one "bucket"
    q_indices = np.round((x - lo) / step)    # which bucket each value falls in (0..levels-1)
    return lo + q_indices * step             # map the bucket index back to a real value

# 4 bits can represent 2**4 = 16 distinct levels.
q4 = quantize(weights, levels=16)
print("\nQuantized to 16 levels (like 4-bit):")
print(np.round(q4, 4))

print("\nAverage rounding error:", np.round(np.mean(np.abs(weights - q4)), 4))
# Expected: the quantized numbers are CLOSE to the originals, with a small error.
# That small error is the price we pay for using way less memory.

**What this does:**

- `weights` is a stand-in for the millions of floats inside a real model.
- `quantize(...)` chops the range into `levels` evenly spaced buckets and snaps each weight to the nearest bucket — exactly the "round to the nearest dollar" idea.
- With `levels=16` we mimic **4-bit** storage (because `2**4 = 16`). Each weight now needs only enough info to name 1 of 16 buckets instead of a full 32-bit float — about **8× less memory**.
- The **average rounding error** is small, which is why a quantized model still works almost as well as the original.

> Real 4-bit quantization (NF4, below) is smarter than this evenly spaced version, but the core idea is identical: **trade a little precision for a lot of memory.**

### ✏️ Exercise

Change `levels` in the `quantize` call to **4** (like 2-bit) and then to **256** (like 8-bit). Print the average rounding error each time. You should see: **fewer levels → bigger error**, **more levels → smaller error**. This is the precision-vs-memory trade-off in one experiment.

In [ ]:
# Your turn:
# for L in [4, 16, 256]:
#     q = quantize(weights, levels=L)
#     print(f"levels={L:4d}  avg error={np.mean(np.abs(weights - q)):.4f}")

## 2. What is QLoRA? (4-bit base + LoRA adapters)

Recall **LoRA** from notebook 11: instead of updating all of a model's weights, you **freeze** the big model and train two small "adapter" matrices per layer. Only those tiny adapters learn; the giant base stays fixed. This already saves a lot of memory and produces a small adapter file.

**QLoRA = LoRA, but the frozen base model is stored in 4-bit.** Two ideas stacked together:

1. **Quantize the base to 4-bit** → the huge frozen model takes ~4× less memory to *hold*.
2. **Train only LoRA adapters on top** → you barely store any gradients or optimizer state.

Why this is a **big deal**:

- A **7B model** that needs ~14 GB in 16-bit now fits in **~3.5 GB** as a 4-bit base.
- The only things you actually *train* are the small adapters — kilobytes to a few megabytes.
- Result: you can fine-tune a **7B-class model on a single free GPU** (like a Colab T4). Before QLoRA, that took expensive multi-GPU servers.

The adapters are trained in higher precision even though the base is 4-bit, so quality stays surprisingly close to a full fine-tune.

![mental model] *Mental model:* the **4-bit base** is a heavy, frozen statue (cheap to store, never changes); the **LoRA adapters** are small sticky notes you attach and write on. You only ever edit the sticky notes.

## 3. The `BitsAndBytesConfig` — telling Transformers to load in 4-bit

To load a model in 4-bit, you build a small config object and pass it to `from_pretrained(...)`. Here's what each setting means. (This cell just *builds* the config object — that part is harmless on any machine; actually *using* it to load a model needs a GPU.)

In [ ]:
from transformers import BitsAndBytesConfig

# This object just describes HOW to quantize. Building it is safe on CPU.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # store the base model's weights in 4-bit
    bnb_4bit_quant_type="nf4",             # use the "NF4" 4-bit format (best for model weights)
    bnb_4bit_use_double_quant=True,        # quantize the quantization constants too -> a bit more memory saved
    bnb_4bit_compute_dtype=torch.bfloat16, # do the actual math in 16-bit for accuracy
)

print(bnb_config)
# Expected: a printout of the config showing the four settings above.
# NOTE: building this config is fine on CPU. USING it (next sections) needs an NVIDIA GPU.

**What this does:** each setting, in plain English:

- **`load_in_4bit=True`** — the headline switch: store the frozen base model's weights in **4-bit** instead of 16/32-bit. This is what makes a 7B model fit in a few GB.
- **`bnb_4bit_quant_type="nf4"`** — picks the **NF4** ("4-bit NormalFloat") format. The buckets aren't evenly spaced like our NumPy demo; they're spaced to match how neural-network weights are actually distributed (clustered near zero), which **loses less accuracy**. It's the recommended default for QLoRA.
- **`bnb_4bit_use_double_quant=True`** — a bonus trick: even the small "scaling constants" used during quantization get quantized too. Saves a little **extra memory** for free. Leave it on.
- **`bnb_4bit_compute_dtype=torch.bfloat16`** — the weights are *stored* in 4-bit, but when the model actually does math it temporarily uses **16-bit (bfloat16)**. This keeps the computation accurate. (Storage precision and compute precision are separate choices.)

Together these four lines are the entire "Q" in QLoRA.

### ✏️ Exercise

Without running anything, predict: if you set `bnb_4bit_use_double_quant=False`, will the model use **more** or **less** memory? (Answer: slightly **more**, because you skip the extra compression of the quantization constants.) Then, build a second `BitsAndBytesConfig` with `load_in_4bit=False` and `load_in_8bit=True` and print it — that's **8-bit** quantization, a middle ground between 4-bit and full precision.

In [ ]:
# Your turn (building configs is CPU-safe):
# config_8bit = BitsAndBytesConfig(load_in_8bit=True)
# print(config_8bit)

## 4. Our dataset: lead-intent classification (instruction style)

We'll reuse the **canonical lead dataset** from earlier notebooks so you can compare techniques fairly. Each "lead" is a synthetic marketing contact with features like family size, income, and what call-to-action they clicked. The label is the **lead intent**: `hot`, `warm`, or `cold`.

For a *generative* model like TinyLlama, we don't attach a classification head — instead we **phrase the task as text**: we write an instruction describing the lead and end the prompt with `"Intent:"`, and we want the model to complete it with ` hot`, ` warm`, or ` cold`.

The cell below regenerates the exact dataset inline (same seed → same data every time).

In [ ]:
import random
random.seed(42)
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
CTAS = ["requested_quote","booked_demo","downloaded_brochure","newsletter_signup"]
CONDITIONS = ["urgent","exploring","just_browsing"]

def make_lead():
    family_size = random.randint(1, 6)
    income = random.choice([25000,40000,55000,70000,85000,100000,120000,150000])
    rent_or_own = random.choice(["rent","own"])
    cta = random.choice(CTAS)
    engagement_month = random.choice(MONTHS)
    current_condition = random.choice(CONDITIONS)
    score = 0
    if cta in ("requested_quote","booked_demo"): score += 2
    elif cta == "downloaded_brochure": score += 1
    if rent_or_own == "own": score += 1
    if income >= 80000: score += 1
    if family_size >= 4: score += 1
    if current_condition == "urgent": score += 2
    elif current_condition == "exploring": score += 1
    lead_intent = "hot" if score >= 5 else ("warm" if score >= 3 else "cold")
    return {"family_size": family_size, "income": income, "rent_or_own": rent_or_own,
            "cta": cta, "engagement_month": engagement_month,
            "current_condition": current_condition, "lead_intent": lead_intent}

leads = [make_lead() for _ in range(200)]
print("Total leads:", len(leads))
print("Example lead:", leads[0])

# Quick look at the label balance:
from collections import Counter
print("Label counts:", Counter(l["lead_intent"] for l in leads))
# Expected: 200 leads, with a mix of hot / warm / cold.

**What this does:**

- `make_lead()` builds one synthetic lead with random features, then applies simple scoring rules to assign a `lead_intent` of `hot`, `warm`, or `cold`. (The rules are deterministic given the features — that's the *pattern* we want the model to learn.)
- `random.seed(42)` makes the 200 leads identical every run, so your results are reproducible.
- `Counter(...)` shows how many of each label we have — useful to confirm we have all three classes.

## 5. Turning leads into instruction prompts

Now we convert each lead dict into a **prompt + answer** pair of text. The prompt describes the lead and ends in `"Intent:"`; the answer is a single word with a leading space (` hot` / ` warm` / ` cold`). That leading space matters — it's how the model naturally continues after a colon.

In [ ]:
def lead_to_prompt(lead):
    """Build the instruction prompt (everything the model SEES) ending in 'Intent:'."""
    return (
        "Classify the sales lead as hot, warm, or cold.\n"
        f"Family size: {lead['family_size']}\n"
        f"Income: {lead['income']}\n"
        f"Rent or own: {lead['rent_or_own']}\n"
        f"Call to action: {lead['cta']}\n"
        f"Engagement month: {lead['engagement_month']}\n"
        f"Current condition: {lead['current_condition']}\n"
        "Intent:"
    )

def lead_to_answer(lead):
    """The target completion: a single word with a LEADING SPACE."""
    return " " + lead["lead_intent"]

# Full training text = prompt + answer (this is what the model learns to produce):
def lead_to_text(lead):
    return lead_to_prompt(lead) + lead_to_answer(lead)

# Show one rendered example:
print("----- PROMPT (what the model sees) -----")
print(lead_to_prompt(leads[0]))
print("----- ANSWER (what it should add) -----")
print(repr(lead_to_answer(leads[0])))
print("----- FULL TRAINING TEXT -----")
print(lead_to_text(leads[0]))

**What this does:**

- `lead_to_prompt` lays out the lead's features as readable lines and **ends with `"Intent:"`** — a natural cue for the model to fill in the class.
- `lead_to_answer` returns the label with a **leading space** (` hot`), matching how text continues after `"Intent:"`. We use `repr(...)` when printing so the space is visible.
- `lead_to_text` glues prompt + answer together. During training the model learns to **produce the whole thing**; at inference we'll give it just the prompt and let it generate the answer.

### ✏️ Exercise

Print the full training text for `leads[5]` and `leads[10]`. Read them out loud — does the assigned `Intent` feel reasonable given the features? Getting a feel for your data by eye is one of the most underrated fine-tuning skills.

In [ ]:
# Your turn:
# print(lead_to_text(leads[5]))
# print(lead_to_text(leads[10]))

## 6. Build a Hugging Face `Dataset` and split it

We wrap our rendered texts in a `datasets.Dataset` (the format `Trainer` expects) and split off a small test set. This part is **CPU-safe** — no GPU needed to prepare data.

In [ ]:
from datasets import Dataset

# Turn the leads into a dataset with one "text" column (prompt+answer).
texts = [lead_to_text(l) for l in leads]
full_ds = Dataset.from_dict({"text": texts})

# Hold out 20% for testing.
split = full_ds.train_test_split(test_size=0.2, seed=42)
train_ds, test_ds = split["train"], split["test"]

print("Train examples:", len(train_ds))
print("Test examples: ", len(test_ds))
print("First train text:\n", train_ds[0]["text"][:120], "...")
# Expected: 160 train / 40 test.

**What this does:**

- `Dataset.from_dict({"text": texts})` creates a Hugging Face dataset with a single `text` column — each row is one full prompt+answer string.
- `train_test_split(test_size=0.2, seed=42)` reserves 20% (40 examples) for evaluation and keeps 160 for training. The seed makes the split reproducible.
- This is the exact same data-prep step you'd use for LoRA or full fine-tuning — **nothing here is QLoRA-specific**.

## 7. 🔴 GPU REQUIRED — Load the base model in 4-bit

Now the QLoRA-specific part. We load **`TinyLlama/TinyLlama-1.1B-Chat-v1.0`** — a small, free, open chat model — **in 4-bit** using our `bnb_config`.

> **This cell needs an NVIDIA GPU.** On CPU/Mac it will detect `HAS_GPU == False` and **skip gracefully** with a message. (If you're on CPU, follow the LoRA fallback in section 12.)

`TinyLlama` is only 1.1B parameters so it's quick to download and train. The *exact same code* works for a 7B model on a bigger GPU — just change the model name.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = None
tokenizer = None

if not HAS_GPU:
    print("⏭️  SKIPPING 4-bit load: no NVIDIA GPU detected.")
    print("    On CPU/Mac, bitsandbytes 4-bit isn't supported.")
    print("    Use the LoRA fallback (notebook 11) instead — see section 12.")
else:
    # Tokenizer first (CPU-safe, but we only need it on the GPU path here).
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    # Causal LMs often have no pad token; reuse the end-of-sequence token as padding.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load the base model in 4-bit using the config from section 3.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,  # <-- the 4-bit instructions
        device_map="auto",              # let accelerate place the model on the GPU
    )
    print("Loaded TinyLlama in 4-bit. 🎉")
    print("Model device:", next(model.parameters()).device)

**What this does (GPU path):**

- `AutoTokenizer.from_pretrained(...)` loads TinyLlama's tokenizer. We set `pad_token = eos_token` because generative models often ship without a padding token, and the `Trainer` needs one to pad batches.
- `AutoModelForCausalLM.from_pretrained(..., quantization_config=bnb_config, device_map="auto")` downloads the model and **loads its weights in 4-bit** per our config. `device_map="auto"` lets `accelerate` put it on the GPU automatically.
- On CPU/Mac the whole block is **skipped** with a friendly message, so the notebook never crashes — it just can't train here.

## 8. 🔴 GPU — `prepare_model_for_kbit_training`

Before attaching LoRA adapters to a quantized model, we run one helper from the `peft` library: **`prepare_model_for_kbit_training(model)`**. ("k-bit" = any low-bit setup like 4-bit or 8-bit.)

In [ ]:
from peft import prepare_model_for_kbit_training

if model is not None:
    # Get the quantized model ready for adapter training.
    model = prepare_model_for_kbit_training(model)
    print("Model prepared for k-bit (4-bit) training. ✅")
else:
    print("⏭️  Skipped (no model loaded — CPU/Mac path).")

**What this does:**

`prepare_model_for_kbit_training` makes a few small but important adjustments so training a **quantized** model is stable:

- **Freezes** all the base (4-bit) weights so only the adapters we add next will train.
- Casts a few sensitive layers (like **layer norms**) back to higher precision for numerical stability.
- Enables **gradient checkpointing** support, which trades a little compute for **less memory** during training.

You don't need to memorize the internals — just remember: **call this once, right after loading a 4-bit/8-bit model and before adding LoRA.**

## 9. 🔴 GPU — Attach LoRA adapters and count trainable params

This is the **"LoRA" in QLoRA**, and it's identical to notebook 11. We describe the adapters with a `LoraConfig`, then wrap the model with `get_peft_model`. Then we print how few parameters we'll actually train.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                                  # rank: size of the small adapter matrices (small = cheap)
    lora_alpha=16,                        # scaling factor for the adapter's effect
    target_modules=["q_proj", "v_proj"],  # which layers to attach adapters to (attention query/value)
    lora_dropout=0.05,                    # a little dropout for regularization
    bias="none",                          # don't train bias terms
    task_type="CAUSAL_LM",                # we're fine-tuning a text-generation (causal) model
)

if model is not None:
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    # Expected (approx): trainable params ~1.1M || all params ~1.1B || trainable% ~0.1%
    # The point: we train a TINY fraction of the model.
else:
    print("⏭️  Skipped (CPU/Mac path). On CPU, use the notebook-11 LoRA classifier instead.")
    print("    The LoraConfig above is exactly what you'd reuse there (minus the 4-bit base).")

**What this does:**

- **`r=8`** — the LoRA **rank**: how big the two little adapter matrices are. Small `r` = fewer trainable params = cheaper. 8 is a common, safe default.
- **`lora_alpha=16`** — a scaling knob controlling how strongly the adapter influences the model. A common rule of thumb is `alpha = 2 * r`.
- **`target_modules=["q_proj", "v_proj"]`** — *which* layers get adapters. `q_proj` and `v_proj` are the attention **query** and **value** projections — the classic, effective choice for transformer LoRA.
- **`task_type="CAUSAL_LM"`** — tells `peft` we're adapting a text-generation model (so it wires up the adapters correctly).
- **`print_trainable_parameters()`** prints something like *"trainable params: 1.1M / 1.1B (~0.1%)"*. That ~0.1% is the whole magic of (Q)LoRA: **train a sliver, freeze the rest.**

### ✏️ Exercise

On the GPU path, change `r` from `8` to `16` and re-run section 9. Watch the **trainable params** number roughly double. Higher rank = more capacity to learn (and more memory). On CPU, just reason about it: doubling `r` doubles the size of the adapter matrices.

## 10. 🔴 GPU — Tokenize and train with `Trainer`

Same `Trainer` loop you've seen before. First we tokenize the `text` column; then we run a **brief** training (just a couple of epochs on 160 tiny examples) so it finishes fast.

For causal-LM fine-tuning, the **labels are the input ids** — the model learns to predict each next token of the full prompt+answer text. A `DataCollatorForLanguageModeling` with `mlm=False` handles that for us.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

if model is not None:
    # 1) Tokenize each text. Keep sequences short — our prompts are small.
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128)

    train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
    test_tok  = test_ds.map(tokenize, batched=True, remove_columns=["text"])

    # 2) Collator: for causal LM, labels = input_ids (mlm=False means "not masked LM").
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    # 3) Training settings (explained in the next markdown cell).
    args = TrainingArguments(
        output_dir="qlora-tinyllama-leads",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        num_train_epochs=2,
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        fp16=False, bf16=True,           # train adapters in bfloat16 (matches our compute dtype)
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=test_tok,
        data_collator=collator,
    )

    trainer.train()
    print("QLoRA training finished. ✅")
else:
    print("⏭️  Skipped (CPU/Mac path). See section 12 for the LoRA fallback you CAN run.")

**What this does:**

- **Tokenize** turns each `text` into `input_ids`; `max_length=128` truncates anything longer (our prompts are well under that).
- **`DataCollatorForLanguageModeling(..., mlm=False)`** batches examples and, for causal LMs, copies `input_ids` into `labels` so the model is trained to predict the next token across the whole prompt+answer.
- **`Trainer`** runs the standard training loop (the same one from notebook 08), but now it's only updating the **LoRA adapters** — the 4-bit base is frozen.
- On CPU/Mac this is skipped; use the fallback in section 12.

### Re-explaining the key hyperparameters (QLoRA context)

- **`learning_rate=2e-4`** — how big each adapter update is. LoRA/QLoRA adapters typically use a **higher** learning rate (1e-4 to 3e-4) than full fine-tuning, because only the small adapters are learning.
- **`num_train_epochs=2`** — how many passes over the data. We keep it tiny for a fast demo; real runs often use 1–3.
- **`per_device_train_batch_size=4`** — examples processed at once. Small, because GPU memory is the constraint even with a quantized base.
- **`gradient_accumulation_steps=2`** — accumulate gradients over 2 mini-batches before updating, giving an **effective batch size of 4 × 2 = 8** without using more memory. This is the standard trick to "fake" a bigger batch on a small GPU.
- **max sequence length (`max_length=128`)** — longer sequences use **a lot** more memory; keep it just long enough to fit your prompts.
- **`warmup_ratio=0.03`** — slowly ramps the learning rate up over the first ~3% of steps so training starts gently and stably.

> Notice: these are the **same** knobs as plain LoRA. QLoRA doesn't add new hyperparameters — it just lets a bigger model fit.

## 11. 🔴 GPU — Save the adapter (and a quick sanity test)

QLoRA's output is **just the small LoRA adapter** — not the whole model. You save the adapter (a few MB), and to use it later you reload the 4-bit base and **attach** the adapter on top. Tiny, portable, shareable.

In [ ]:
if model is not None:
    # Save ONLY the LoRA adapter weights (small!).
    model.save_pretrained("qlora-tinyllama-leads-adapter")
    tokenizer.save_pretrained("qlora-tinyllama-leads-adapter")
    print("Adapter saved to ./qlora-tinyllama-leads-adapter")

    # Quick sanity check: feed a test PROMPT and see what the model generates.
    sample = test_ds[0]["text"]
    prompt = sample.rsplit("Intent:", 1)[0] + "Intent:"   # cut off the gold answer
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=3, do_sample=False)
    generated = tokenizer.decode(out[0], skip_special_tokens=True)
    print("\n--- PROMPT ---\n", prompt)
    print("--- MODEL CONTINUATION ---\n", generated[len(prompt):])
    print("--- GOLD ANSWER ---", sample.rsplit("Intent:", 1)[1])
    # Expected: after brief training the model often produces ' hot'/' warm'/' cold'.
    # It won't be perfect after only 2 epochs on tiny data — that's fine for a demo.
else:
    print("⏭️  Skipped (CPU/Mac path).")

**What this does:**

- **`save_pretrained("...-adapter")`** writes only the LoRA adapter weights (plus a tiny config). This is what you'd upload to the Hub or hand to a teammate — a few megabytes, not gigabytes.
- For the sanity check we take a test example, **strip off the gold answer** after `"Intent:"`, and let the model **generate** the next few tokens. We compare its continuation to the real label.
- `do_sample=False` makes generation **deterministic** (always picks the most likely token) so the demo is repeatable.
- After only 2 epochs on 160 tiny examples it may not be perfect — the goal here is to see the full pipeline run, not to win a benchmark.

> To **reload** later: load the 4-bit base with `from_pretrained(MODEL_NAME, quantization_config=bnb_config)`, then `PeftModel.from_pretrained(base, "qlora-tinyllama-leads-adapter")`.

## 12. 🟡 CPU / Mac fallback — what to do without an NVIDIA GPU

> ### 🟡 No NVIDIA GPU? Here's your plan.
> The 4-bit cells above need CUDA, so on a **CPU-only laptop** or **Apple Silicon Mac** they skip themselves. To still get hands-on with adapter fine-tuning:
>
> 1. **Open notebook 11 (LoRA)** and run its **LoRA classifier** on this same lead dataset. LoRA (without 4-bit quantization) runs fine on CPU with a small model — it's the *exact same adapter idea*, just with a normal-precision base.
> 2. Everything you learned here transfers directly: the `LoraConfig`, `target_modules`, `get_peft_model`, the `Trainer` loop, and `save_pretrained` are **identical**. The only thing you drop is the `BitsAndBytesConfig` / 4-bit loading.
> 3. When you later get access to a GPU (e.g. a **free Colab T4**: *Runtime → Change runtime type → T4 GPU*), come back and run this notebook top to bottom unchanged.
>
> **Bottom line:** QLoRA = LoRA + 4-bit base. If you can't do the 4-bit part, do plain LoRA — you lose the big-model-on-small-GPU superpower, but you learn the same workflow.

The cell below confirms which path you're on and points you to the fallback if needed.

In [ ]:
if HAS_GPU:
    print("✅ You have a GPU — you ran the full QLoRA recipe above.")
else:
    print("🟡 No GPU detected. Fallback plan:")
    print("   1. Open 11_lora_finetuning.ipynb")
    print("   2. Run its LoRA classifier on the SAME lead dataset (CPU-friendly).")
    print("   3. Reuse the LoraConfig / Trainer steps from this notebook unchanged.")
    print("   4. Revisit this notebook on a free Colab T4 GPU when you can.")

## 13. When to choose QLoRA vs. LoRA vs. full fine-tuning

A quick decision guide. All three produce a fine-tuned model; they differ in **memory**, **cost**, and **how much you can change**.

| Method | What trains | Base stored as | Memory need | Use when... |
|---|---|---|---|---|
| **Full fine-tuning** | **All** weights | 16/32-bit | Very high (many×model size) | You have lots of GPUs *and* lots of data, and need maximum quality / big behavior changes. |
| **LoRA** | Small adapters | 16-bit (full) | Moderate | The model fits in memory at 16-bit and you want cheap, fast, swappable fine-tunes. Great on CPU for small models. |
| **QLoRA** | Small adapters | **4-bit** (quantized) | **Low** | The 16-bit base is **too big** for your GPU. QLoRA shrinks the base so a large model fits on a single small GPU. Needs an NVIDIA GPU. |

**Rules of thumb:**

- **Model fits comfortably at 16-bit on your hardware?** → Use **LoRA** (simpler, no quantization, works on CPU for small models).
- **Model is too big to fit at 16-bit, but you have *one* NVIDIA GPU?** → Use **QLoRA** (the whole reason it exists).
- **You have a GPU cluster, lots of data, and need the absolute best?** → Consider **full fine-tuning**.

For most beginners on free Colab wanting to fine-tune a 7B model, **QLoRA is the answer.**

### ✏️ Exercise

For each scenario, pick LoRA, QLoRA, or full fine-tuning:

1. Fine-tuning `distilgpt2` (82M params) on your laptop CPU.
2. Fine-tuning a 7B model on one free Colab T4 (16 GB).
3. A company with 8× A100 GPUs building a flagship model on millions of examples.

*(Answers: 1 → LoRA, 2 → QLoRA, 3 → full fine-tuning.)*

## Common mistakes & how to debug them

- **`ImportError: ... bitsandbytes` or "4-bit not supported".** You're on CPU or Apple Silicon, or `bitsandbytes` isn't installed. 4-bit needs an **NVIDIA GPU**. Use the LoRA fallback (section 12) or switch to a Colab T4.
- **CUDA out of memory (OOM).** Lower `per_device_train_batch_size`, raise `gradient_accumulation_steps` to compensate, shorten `max_length`, or pick a smaller model. Quantization helps a lot but the *activations* during training still use memory.
- **Forgetting `prepare_model_for_kbit_training`.** Skipping it on a 4-bit model leads to unstable training or errors. Always call it right after loading and before `get_peft_model`.
- **No pad token.** Causal LMs often have `tokenizer.pad_token = None`, which crashes batching. Fix with `tokenizer.pad_token = tokenizer.eos_token` (we did this in section 7).
- **Saving the whole model instead of the adapter.** `model.save_pretrained(...)` on a PEFT model saves **only the adapter** — that's correct and intended. To run it later you reload the 4-bit base and attach the adapter.
- **Expecting perfect accuracy from a tiny demo.** 2 epochs on 160 examples won't max out a benchmark. To improve: more/cleaner data, more epochs, or a higher LoRA `r`.
- **Wrong `target_modules` names.** They must match the model's actual layer names. For Llama-family models `["q_proj","v_proj"]` is right; other architectures may use different names (e.g. `["query","value"]`).

## Summary

- **Quantization** stores model weights in **fewer bits** (e.g. 4-bit) to save lots of memory at a small accuracy cost — like rounding prices to the nearest dollar. We saw the idea in a CPU NumPy demo.
- **QLoRA = a 4-bit quantized frozen base model + small trainable LoRA adapters.** It lets you fine-tune a **7B-class model on a single small GPU** (even a free Colab T4).
- **`BitsAndBytesConfig`** turns on 4-bit: `load_in_4bit`, `bnb_4bit_quant_type="nf4"`, `bnb_4bit_use_double_quant`, and `bnb_4bit_compute_dtype=torch.bfloat16`.
- The recipe: **load 4-bit → `prepare_model_for_kbit_training` → `LoraConfig` + `get_peft_model` → tokenize → `Trainer` → save the adapter.** Everything except the 4-bit load is the same LoRA workflow from notebook 11.
- Hyperparameters (LR, epochs, batch size, gradient accumulation, sequence length, warmup) are the **same as LoRA** — QLoRA adds memory savings, not new knobs.
- **Decision guide:** model fits at 16-bit → **LoRA**; too big but one GPU → **QLoRA**; cluster + max quality → **full fine-tuning**.
- **No NVIDIA GPU?** Use the LoRA fallback (notebook 11) — you learn the identical workflow minus the 4-bit trick.

## What to learn next

Next up: **`13_evaluation.ipynb`**. You can now *train* fine-tuned models with LoRA and QLoRA — but how do you know if they're actually any **good**? The next lesson covers **evaluating** fine-tuned models: measuring accuracy and other metrics, building a fair test set, comparing the fine-tuned model against the base model, and spotting overfitting. Training is only half the job; evaluation tells you whether it worked.